Repo URL: https://github.com/<redacted>/assignment-4-equation-of-a-slime-<redacted>  
Commit ID: 89a4774db790bcada381f52789043c4997b9bc67

In [1]:
# Imports section
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
from sklearn.preprocessing import PolynomialFeatures

## Part 1. Loading the dataset

In [2]:
# Using pandas load the dataset (load remotely, not locally)
slime_data = pd.read_csv("science_data_large.csv")

# Output the first 15 rows of the data
print(f"Slime data (15 rows):\n{slime_data[:15].to_string()}\n")

# Display a summary of the table information (number of datapoints, etc.)
print(f"Slime data summary:\n{slime_data.describe()}\n")

Slime data (15 rows):
    Temperature °C  Mols KCL     Size nm^3
0              469       647  6.244743e+05
1              403       694  5.779610e+05
2              302       975  6.196847e+05
3              779       916  1.460449e+06
4              901        18  4.325726e+04
5              545       637  7.124634e+05
6              660       519  7.006960e+05
7              143       869  2.718260e+05
8               89       461  8.919803e+04
9              294       776  4.770210e+05
10             991       117  2.441771e+05
11             307       781  5.006455e+05
12             206        70  3.145200e+04
13             437       599  5.390215e+05
14             566        75  9.185271e+04

Slime data summary:
       Temperature °C     Mols KCL     Size nm^3
count     1000.000000  1000.000000  1.000000e+03
mean       500.500000   471.530000  5.086111e+05
std        288.819436   288.482872  4.474838e+05
min          1.000000     1.000000  1.611429e+01
25%        250.750000   

## Part 2. Splitting the dataset

In [3]:
# Take the pandas dataset and split it into our features (X) and label (y)
slime_X = slime_data.drop("Size nm^3", axis=1)
slime_y = slime_data["Size nm^3"]

# Use sklearn to split the features and labels into a training/test set. (90% train, 10% test)
X_train, X_test, y_train, y_test = train_test_split(slime_X, slime_y, test_size=0.1, random_state=42)

## Part 3. Perform a Linear Regression

In [4]:
# Use sklearn to train a model on the training set
slime_linear_model = LinearRegression()
slime_linear_model.fit(X_train, y_train)

# Create a sample datapoint and predict the output of that sample with the trained model
sample_input = X_test.iloc[[1]]
actual_output = y_test.iloc[1]
predicted_output = slime_linear_model.predict(sample_input)

print(f"Predicted size of slime exposed to {sample_input.iloc[0, 0]}°C and {sample_input.iloc[0, 1]} Mols of KCL is {predicted_output[0]} (as compared to actual value of {actual_output})")

# Report on the score for that model, in your own words (markdown, not code) explain what the score means
print(f"Score for model: {slime_linear_model.score(X_test, y_test)}")

# Extract the coefficents and intercept from the model and write an equation for your h(x) using LaTeX
print(f"h(x) = {slime_linear_model.coef_[0]}x_1 + {slime_linear_model.coef_[1]}x_2 + {slime_linear_model.intercept_}")

Predicted size of slime exposed to 635°C and 668 Mols of KCL is 830451.797320274 (as compared to actual value of 868729.2571)
Score for model: 0.8552472077276095
h(x) = 866.1464133719209x_1 + 1032.6950664857964x_2 + -409391.47958340764


### Scoring the model
$R^2 = 0.85524$  
The model scores a 0.85524. This score, also known as the coefficient of determination, describes how well the model predicts the variation of the model from the mean. A score of 1 (the highest possible) indicates a perfect predictor of variation from the mean, while a 0 or less indicates predictive power no better (or even worse) than simply always predicting the mean. This model's score is fairly close to 1, meaning it is a solid predictor of the variation of the dataset.

### Model equation
$h(x) = -409390 + 866.15x_1 + 1032.7x_2$

## Part 4. Use Cross Validation

In [5]:
# Use the cross_val_score function to repeat your experiment across many shuffles of the data
kf = KFold(n_splits=10, random_state=42, shuffle=True)
scores = cross_val_score(slime_linear_model, X_train, y_train, cv=kf)

print(scores)

# Report on their finding and their significance

[0.85415512 0.87745252 0.82844906 0.85256016 0.81828206 0.86383763
 0.85056956 0.86539158 0.88476697 0.87350793]


### Cross-validation scores  
The scores of 10-fold cross-validation were found to range from 0.81828 to 0.88477.  
All of these scores are quite good (near 1), so they all act as good predictive models. They are also all similar in score (a difference of only 0.06648 between the highest and lowest scores). This analysis shows that the models trained on different folds of the training set are consistent in their strength, which helps validate that the model is generalizing well across the training set and not overfitting. This means we can be reasonably sure that a final model trained on the whole training set will generalize well without overfitting on data outside the training set (e.g. the test set).

## Part 5. Using Polynomial Regression

In [6]:
# Using the PolynomialFeatures library perform another regression on an augmented dataset of degree 2
poly = PolynomialFeatures(2)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

slime_poly_model = LinearRegression()
slime_poly_model.fit(X_train_poly, y_train)

# Report on the metrics and output the resultant equation as you did in Part 3.
print(f"Score for model: {slime_poly_model.score(X_test_poly, y_test)}")
print(f"h(x) = {slime_poly_model.intercept_} + {slime_poly_model.coef_[1]}x_1 + {slime_poly_model.coef_[3]}x_1^2 + {slime_poly_model.coef_[2]}x_2 + {slime_poly_model.coef_[5]}x_2^2 + {slime_poly_model.coef_[4]}x_1x_2")

poly_scores = cross_val_score(slime_poly_model, X_train_poly, y_train, cv=kf)
print(poly_scores)

Score for model: 1.0
h(x) = 2.0479375962167978e-05 + 11.999999977691196x_1 + 1.2645884339690383e-11x_1^2 + -1.2719708848869804e-07x_2 + 0.028571428708642932x_2^2 + 2.0000000000172227x_1x_2
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


### Scoring the model
$R^2 = 1$  
The model scores a perfect 1. A score of 1 is the highest possible, indicating that the model is a perfect predictor of variation from the mean. Even with cross-validation, each of the ten folds score a 1. This could be a sign of overfitting, but further testing on separate data would be required to confirm or dispel this possibility. If the model is not simply overfitted, a perfect score of 1 means that the function that the model represents perfectly describes the underlying relationship of the features to the label and will be likely to predict very accurately on untrained data.

### Model equation
$h(x) = 0.00002 + 12.000x_1 + 0.02857x_2^2 + 2.0000x_1x_2$

The coefficients for $x_1^2$ and $x_2$ are zero when rounded to 5 significant figures and are omitted from the above function